# RNA candidate workflow: an executable walkthrough

**All coordinates and targets generated below are synthetic.** They demonstrate data handling, candidate selection, submission validation, and rigid segment assembly. They are not biological folding predictions, recovered competition submissions, or evidence of a leaderboard score.

Run cells from top to bottom after installing the repository requirements in this notebook's Python environment. NumPy is required; the final plot is optional. This notebook works when launched from the repository root or its `notebooks/` folder. It does not download datasets or neural-network weights.


In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import sys
import tempfile

import numpy as np

HERE = Path.cwd().resolve()
ROOT = next((path for path in (HERE, *HERE.parents)
             if (path / "rna_folding" / "sequence.py").is_file()
             and (path / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Launch the notebook from the repository root or notebooks folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rna_folding.sequence import load_sequences, route_model, segment_ranges
from rna_folding.geometry import kabsch_align, stitch_segments
from rna_folding.io import validate_submission
from examples.inspect_sequences import summarize_sequences

# Keep each run separate and reproducible; no source files are overwritten.
(ROOT / "outputs").mkdir(exist_ok=True)
WORK = Path(tempfile.mkdtemp(prefix="notebook_", dir=ROOT / "outputs"))
print("Repository:", ROOT)
print("Notebook outputs:", WORK)


def run_cli(*arguments):
    result = subprocess.run(
        [sys.executable, "-m", "rna_folding", *map(str, arguments)],
        cwd=ROOT, capture_output=True, text=True,
    )
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print(result.stderr.strip())
    result.check_returncode()
    return result


## 1. Run the CPU pipeline demonstration

The CLI builds artificial candidate coordinates, selects five candidates for each synthetic target, and checks the submission schema. This exercises the local workflow; it does not run Protenix, DRfold2, or trRNA inference.


In [ ]:
DEMO = WORK / "demo"
run_cli("demo", "--output-dir", DEMO)
print("Generated files:")
for path in sorted(DEMO.rglob("*")):
    if path.is_file() and path.suffix in {".json", ".csv"}:
        print(" ", path.relative_to(DEMO))


## 2. Inspect the plan, selection audit, and submission

A plan assigns explicit length buckets. The audit explains how the available candidates were selected. The submission contains one row per residue and fifteen coordinate columns: three coordinates for each of five conformers. Validation confirms format and coverage, not biological accuracy.


In [ ]:
SEQUENCES = DEMO / "sequences.csv"
SUBMISSION = DEMO / "submission.csv"
PLAN = DEMO / "plan.json"
AUDIT = DEMO / "audit.json"

records = load_sequences(SEQUENCES)
for label, path in [("Plan", PLAN), ("Selection audit", AUDIT)]:
    value = json.loads(path.read_text(encoding="utf-8"))
    print(label + ":")
    print(json.dumps(value, indent=2)[:5000])

with SUBMISSION.open(newline="", encoding="utf-8") as stream:
    reader = csv.DictReader(stream)
    print("Submission columns:", reader.fieldnames)
    first_three = [row for _, row in zip(range(3), reader)]
print("First three residue rows:")
print(json.dumps(first_three, indent=2))
print("Validation:", validate_submission(SUBMISSION, records))


## 3. Inspect sequence statistics

The same EDA helper works with a real sequence CSV that you obtain separately. It reads `target_id` and `sequence` only and never opens structure labels. Here its statistics describe the artificial demo targets. Base proportions are calculated across all residues, so longer sequences contribute more bases.


In [ ]:
summary = summarize_sequences(records)
print(json.dumps(summary, indent=2))
assert sum(summary["base_counts"].values()) == summary["total_residues"]
assert sum(summary["route_counts"].values()) == summary["n_targets"]
assert np.isclose(sum(summary["base_proportions"].values()), 1.0)

# The standalone command for your own data (run from the repository root) is:
# python -m examples.inspect_sequences --sequences data/test_sequences.csv \
#     --output outputs/sequence_summary.json


## 4. See the routing boundaries

The historical notes overlap at 100 nt. This reconstruction makes that boundary explicit: 1–100, 101–480, and >480 nt. The `trrosettarna` routing key is the code's label for the notes' short-sequence trRNA branch; the exact historical model identity/version remains unverified. These buckets express a configurable workflow choice, not proven optimal model assignments.


In [ ]:
for length in [1, 99, 100, 101, 479, 480, 481, 600]:
    print(f"{length:>3} nt -> {route_model(length)}")

windows = segment_ranges(600, window=300, overlap=50)
print("600-nt windows, 0-based half-open:", windows)
coverage = np.zeros(600, dtype=int)
for start, stop in windows:
    coverage[start:stop] += 1
assert np.all(coverage >= 1)
print("Covered residues:", np.count_nonzero(coverage))


## 5. Recover a synthetic curve from independently rotated segments

A prediction segment can be arbitrarily rotated and translated. Joining raw coordinates would give discontinuities. The package instead finds a proper rigid transform from matching overlap residues and blends their aligned coordinates.

This example deliberately cuts one known mathematical curve into overlapping segments, then moves each segment. Recovering that same curve verifies the geometry operation. It cannot demonstrate recovery of the unknown long-range contacts in real RNA or the accuracy of independently folded segments.


In [ ]:
length = 90
parameter = np.linspace(0.0, 5.0 * np.pi, length)
curve = np.column_stack([
    8.0 * np.cos(parameter),
    8.0 * np.sin(parameter),
    0.7 * np.arange(length),
])
windows = segment_ranges(length, window=40, overlap=15)
segments = []
for index, (start, stop) in enumerate(windows):
    angle = 0.7 * index
    rotation = np.array([
        [np.cos(angle), -np.sin(angle), 0.0],
        [np.sin(angle), np.cos(angle), 0.0],
        [0.0, 0.0, 1.0],
    ])
    translation = np.array([25.0 * index, -11.0 * index, 9.0 * index])
    segments.append((start, curve[start:stop] @ rotation + translation))

assembled = stitch_segments(segments, length)
aligned = kabsch_align(assembled, curve)
rmsd = float(np.sqrt(np.mean(np.sum((aligned - curve) ** 2, axis=1))))
print("Segment ranges:", windows)
print(f"Synthetic reconstruction RMSD: {rmsd:.3e} coordinate units")
assert rmsd < 1e-8
print("This is a geometry check, not TM-score or a competition result.")


## 6. Optional 3D illustration

Matplotlib is only needed for this cell. If it is unavailable, the numerical demonstration above still completes.


In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Matplotlib is unavailable; skipping the optional plot.")
else:
    figure = plt.figure(figsize=(10, 4))
    left = figure.add_subplot(121, projection="3d")
    right = figure.add_subplot(122, projection="3d")
    for start, coordinates in segments:
        left.plot(*coordinates.T, label=f"start {start}")
    left.set_title("Synthetic segments before alignment")
    left.legend(fontsize=8)
    right.plot(*curve.T, label="Known mathematical curve", linewidth=2)
    right.plot(*aligned.T, "--", label="Reassembled curve")
    right.set_title("Synthetic overlap reconstruction")
    right.legend(fontsize=8)
    for axis in (left, right):
        axis.set_xlabel("x (synthetic units)")
        axis.set_ylabel("y (synthetic units)")
        axis.set_zlabel("z (synthetic units)")
    figure.tight_layout()
    plt.show()
    plt.close(figure)


## Using the workflow with your own competition files

1. Obtain sequence data through its authorized source, then inspect it with `examples.inspect_sequences`.
2. Generate predictions using an upstream model environment, or import genuine candidate outputs using the repository's documented input format.
3. Retain model, checkpoint, seed, target, and scoring provenance before candidate selection and submission export.
4. Evaluate against separately held-out reference structures with the documented USalign helper. Keep reference information outside the inference and candidate-selection workflow.

The original report and solution-note scores belong to historical documentation. This notebook creates no replacement leaderboard results. For the file schemas and model setup, see the repository README and `docs/`.
